# Injury Rates: Resident and Patient Handling per 100 Full-Time Workers

The other modules count injuries. Counts cannot compare a large hospital with a small nursing home, or one state with another. This module divides injuries by hours worked, using the hours each establishment reports on OSHA Form 300A, and states rates per 100 full-time-equivalent workers (200,000 hours), the standard OSHA convention.

**Data** OSHA Injury Tracking Application, 2025 (submissions through 2026-08-31): case detail (Forms 300/301) and establishment summaries (Form 300A)
**Settings** Hospitals (NAICS 622) and nursing care facilities (NAICS 6231)
**Source data** US Government work, 17 U.S.C. §105

### Who is counted

Case detail is required only of establishments with 100 or more employees in designated industries. Rates therefore cover establishments that (a) reported 100 or more average employees on Form 300A and (b) either submitted case detail or reported no recordable cases. Establishments whose 300A shows recordable cases but which submitted no case detail are excluded, because their handling injuries cannot be counted; how many were excluded is reported.

### Classification

Handling cases use the frozen rule from `nursing-facility-handling/`; lateral transfers use the final task rule from `handling-task-types/`. Both were validated by blind review. Raw rule rates are reported alongside rates adjusted by the measured precision (hospital handling 0.80, nursing-facility handling 0.89, hospital lateral 0.83).

## 1. Environment and data

In [1]:
!pip install -q duckdb pandas requests
import os, re, requests, duckdb, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
def fetch(url, path):
    if not os.path.exists(path):
        with requests.get(url, headers=HDRS, stream=True, timeout=1800) as r:
            r.raise_for_status()
            with open(path, 'wb') as f:
                for ch in r.iter_content(1 << 20): f.write(ch)
    print(f'{path}: {os.path.getsize(path)/1e6:.0f} MB')
fetch('https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_08-31-2026.csv', 'data/cases.csv')
fetch('https://www.osha.gov/sites/default/largefiles/ITA_300A_Summary_Data_2025_through_08-31-2026.csv', 'data/300a.csv')
con = duckdb.connect(':memory:')
for t, p in [('cases', 'data/cases.csv'), ('s300a', 'data/300a.csv')]:
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('{p}', all_varchar=true, ignore_errors=true, sample_size=-1)")
def resolver(t):
    cols = {c.lower(): c for c in [r[0] for r in con.execute(f'DESCRIBE {t}').fetchall()]}
    return lambda *n: next(cols[x.lower()] for x in n if x.lower() in cols)
cc, sc = resolver('cases'), resolver('s300a')
print(con.execute('SELECT COUNT(*) FROM cases').fetchone()[0], 'cases;', con.execute('SELECT COUNT(*) FROM s300a').fetchone()[0], '300A rows')

data/cases.csv: 475 MB
data/300a.csv: 88 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

782582 cases; 400288 300A rows


## 2. Establishments and hours (Form 300A)

In [2]:
SETTING = "CASE WHEN {n} LIKE '6231%' THEN 'nursing facility' WHEN {n} LIKE '622%' THEN 'hospital' END"
est = con.execute(f'''
SELECT "{sc('establishment_id')}" AS establishment_id,
       ANY_VALUE({SETTING.format(n='"' + sc('naics_code') + '"')}) AS setting,
       ANY_VALUE("{sc('state')}") AS state,
       MAX(TRY_CAST("{sc('annual_average_employees')}" AS INT)) AS employees,
       MAX(TRY_CAST("{sc('total_hours_worked')}" AS BIGINT)) AS hours,
       MAX(COALESCE(TRY_CAST("{sc('total_deaths')}" AS INT), 0) + COALESCE(TRY_CAST("{sc('total_dafw_cases')}" AS INT), 0)
         + COALESCE(TRY_CAST("{sc('total_djtr_cases')}" AS INT), 0) + COALESCE(TRY_CAST("{sc('total_other_cases')}" AS INT), 0)) AS recordable_300a
FROM s300a WHERE "{sc('naics_code')}" LIKE '6231%' OR "{sc('naics_code')}" LIKE '622%'
GROUP BY 1''').df()
est = est[(est.hours > 0) & (est.employees >= 100)].copy()
# Hours plausibility. Some 300A records report hours far out of line with the employee count
# (for example, a whole company's hours on one facility's form). A full-time worker works about
# 2,000 hours a year; part-time staff lower the average. Records outside 500-4,000 hours per
# average employee are flagged and excluded from rates; results without this filter are also reported.
est['hours_per_employee'] = est.hours / est.employees
est['hours_plausible'] = est.hours_per_employee.between(500, 4000)
print(est.groupby('setting').agg(establishments=('establishment_id', 'size'), hours=('hours', 'sum'), implausible_hours=('hours_plausible', lambda s: int((~s).sum()))))
print(est.groupby('setting').hours_per_employee.describe(percentiles=[.01, .05, .5, .95, .99]).round(0))

                  establishments        hours  implausible_hours
setting                                                         
hospital                    4845  12354763720                108
nursing facility            5365   9560840673                341
                   count     mean       std  min     1%      5%     50%     95%       99%         max
setting                                                                                              
hospital          4845.0   2773.0   37839.0  0.0  160.0  1019.0  1515.0  2233.0    2920.0   2094953.0
nursing facility  5365.0  11510.0  260850.0  0.0   79.0   722.0  1518.0  2405.0  154410.0  11733928.0


## 3. Cases, classified

Same frozen rules as the other modules.

In [3]:
TASKS = [
 ('lateral transfer between surfaces',
  r"\b(?:lateral(?:ly)? (?:transfer\w*|slid\w*|mov\w*)|bed to bed|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:operating|or|surgical|procedure|exam|examination|ct|cat scan|mri|x-?ray|imaging|radiology|cath\w*|ir|endoscopy|dialysis) (?:room )?(?:table|bed)|"
  r"(?:operating|or|surgical|procedure|exam|ct|mri|x-?ray|imaging) table to|"
  r"(?:bed|table|stretcher|gurney|cart|litter|cot) to (?:a |an |the |another |new |hospital |ems |shower |her |his )?(?:bed|table|stretcher|gurney|cart|litter|cot)|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:stretcher|gurney|litter|cot)\b|(?:stretcher|gurney|litter|cot) to|"
  r"slid\w* (?:the )?(?:resident|patient|pt|him|her) (?:over|across|onto|from))\b"),
 ('recovery from the floor',
  r"\b(?:off (?:of )?the (?:floor|ground)|from the (?:floor|ground)|up off|(?:found|was|lying) on the floor|lift\w* (?:\w+ ){0,2}(?:from|off) (?:the )?floor)\b"),
 ('catching or supporting a falling person',
  r"\b(?:catch\w*|caught (?:the |a )?(?:resident|patient|him|her)|began to fall|started to fall|was falling|"
  r"prevent\w* (?:\w+ ){0,2}from falling|keep \w+ from falling|lower\w* (?:\w+ ){0,2}to the (?:floor|ground)|(?:legs|knees) (?:gave out|buckled))\b"),
 ('seated or standing transfer',
  r"\b(?:(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair) to (?:a |the )?(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair)|"
  r"sit[- ]to[- ]stand|stand[- ]pivot|pivot\w*|into (?:the |a |her |his )?(?:wheel ?chair|chair|recliner|w/?c)|out of (?:the |a |her |his )?(?:wheel ?chair|chair|bed|recliner)|"
  r"on(?:to)? (?:the )?(?:toilet|commode)|off (?:the )?(?:toilet|commode)|(?:to|help\w* \w+) stand)\b"),
 ('repositioning in bed',
  r"\b(?:pull\w* (?:\w+ ){0,3}up (?:in|to)|boost\w*|reposition\w*|turn\w* (?:\w+ ){0,3}(?:in bed|over|on (?:her|his|their|the) side)|"
  r"roll\w* (?:the |a )?(?:resident|patient|pt|him|her)|draw ?sheet|turning sheet|up in bed|head of (?:the )?bed)\b"),
 ('ambulation', r"\b(?:ambulat\w*|walking (?:the |a |with (?:the |a )?)?(?:resident|patient|pt))\b"),
 ('transfer, type not stated', r"\btransfer\w*\b"),
]
SEATED_TO_STRETCHER = r"\b(?:wheel ?chair|chair|w/?c|recliner|geri ?chair) to (?:a |an |the )?(?:\w+ )?(?:stretcher|gurney|cot|table)\b"
def task(s):
    if re.search(SEATED_TO_STRETCHER, s): return 'seated or standing transfer'
    for name, p in TASKS:
        if re.search(p, s): return name
    return 'handling, task not stated'

PERSON   = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE   = r"patient handling|resident handling|resident care transfer"
nar = "lower(concat_ws(' ', " + ', '.join(f'"{cc(c)}"' for c in
      ['new_nar_before_incident', 'new_nar_what_happened', 'new_nar_injury_illness', 'new_nar_object_substance']) + '))'
cs = con.execute(f"""SELECT "{cc('establishment_id')}" AS establishment_id, {nar} AS narr,
       {SETTING.format(n='"' + cc('naics_code') + '"')} AS setting,
       CASE WHEN TRY_CAST("{cc('dafw_num_away')}" AS INT) >= 0 THEN TRY_CAST("{cc('dafw_num_away')}" AS INT) END AS dafw,
       CASE WHEN TRY_CAST("{cc('djtr_num_tr')}" AS INT) >= 0 THEN TRY_CAST("{cc('djtr_num_tr')}" AS INT) END AS djtr
     FROM cases WHERE "{cc('naics_code')}" LIKE '6231%' OR "{cc('naics_code')}" LIKE '622%' """).df()
n = cs.narr.fillna('')
cs['handling'] = (n.str.contains(PERSON, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)
cs['lateral'] = False
cs.loc[cs.handling, 'lateral'] = n[cs.handling].map(task).eq('lateral transfer between surfaces')
cs['lost_time'] = cs.dafw.fillna(0) > 0
per = cs.groupby('establishment_id').agg(cases=('narr', 'size'), handling=('handling', 'sum'), lateral=('lateral', 'sum'),
                                          handling_lost_time=('handling', lambda s: int((s & cs.loc[s.index, 'lost_time']).sum())),
                                          lateral_lost_time=('lateral', lambda s: int((s & cs.loc[s.index, 'lost_time']).sum())))
print(len(cs), 'cases in hospitals and nursing facilities')

207550 cases in hospitals and nursing facilities


## 4. Population for rates

Establishments with 100+ employees and hours on Form 300A. Those whose 300A reports recordable cases but which have no case-detail records are excluded.

Records with implausible hours (outside 500–4,000 hours per average employee) are also excluded. This filter was added after the first run showed nursing-facility hours inconsistent with employee counts in several states; rates without it are reported in Section 5b.

In [4]:
df = est.merge(per, on='establishment_id', how='left').fillna({'cases': 0, 'handling': 0, 'lateral': 0, 'handling_lost_time': 0, 'lateral_lost_time': 0})
df['missing_detail'] = (df.recordable_300a > 0) & (df.cases == 0)
cov = df.groupby('setting').agg(establishments=('establishment_id', 'size'), excluded_missing_detail=('missing_detail', 'sum'),
                                excluded_implausible_hours=('hours_plausible', lambda s: int(((~s) & ~df.loc[s.index, 'missing_detail']).sum())))
cov['included'] = cov.establishments - cov.excluded_missing_detail - cov.excluded_implausible_hours
cov.to_csv('derived/coverage.csv'); print(cov)
pop = df[~df.missing_detail & df.hours_plausible].copy()
pop_nofilter = df[~df.missing_detail].copy()

                  establishments  excluded_missing_detail  excluded_implausible_hours  included
setting                                                                                        
hospital                    4845                      811                          76      3958
nursing facility            5365                     1559                         210      3596


## 5. Rates per 100 full-time workers

Pooled rate = total cases × 200,000 ÷ total hours. Adjusted rates multiply by the measured precision of the rule.

In [5]:
PREC = {('hospital', 'handling'): 0.80, ('nursing facility', 'handling'): 0.893, ('hospital', 'lateral'): 0.833}
def rates(g, by):
    a = g.groupby(by, observed=True).agg(establishments=('establishment_id', 'size'), hours=('hours', 'sum'), all_cases=('cases', 'sum'),
                          handling=('handling', 'sum'), lateral=('lateral', 'sum'),
                          handling_lost_time=('handling_lost_time', 'sum'), lateral_lost_time=('lateral_lost_time', 'sum'))
    for k in ['all_cases', 'handling', 'lateral', 'handling_lost_time']:
        a[f'{k}_rate'] = (a[k] * 200000 / a.hours).round(3)
    return a
by_setting = rates(pop, 'setting')
for (s, k), p in PREC.items():
    if s in by_setting.index: by_setting.loc[s, f'{k}_rate_adjusted'] = round(by_setting.loc[s, f'{k}_rate'] * p, 3)
by_setting.to_csv('derived/rates-by-setting.csv'); by_setting.T

setting,hospital,nursing facility
establishments,3.958000e+03,3.596000e+03
hours,7.562131e+09,1.039777e+09
all_cases,1.782910e+05,2.273800e+04
handling,2.035500e+04,4.951000e+03
lateral,1.703000e+03,2.100000e+01
handling_lost_time,8.371000e+03,1.803000e+03
lateral_lost_time,7.360000e+02,4.000000e+00
all_cases_rate,4.715000e+00,4.374000e+00
handling_rate,5.380000e-01,9.520000e-01
lateral_rate,4.500000e-02,4.000000e-03


### 5b. Sensitivity: without the hours filter

In [6]:
nofilter = rates(pop_nofilter, 'setting')
nofilter.to_csv('derived/rates-by-setting-no-hours-filter.csv')
pd.DataFrame({'with filter': by_setting.handling_rate, 'without filter': nofilter.handling_rate, 'all cases, with filter': by_setting.all_cases_rate, 'all cases, without filter': nofilter.all_cases_rate})

,with filter,without filter,"all cases, with filter","all cases, without filter"
setting,,,,
hospital,0.538,0.468,4.715,4.114
nursing facility,0.952,0.177,4.374,0.815


### By state (states with at least 10 establishments in the setting)

In [7]:
by_state = rates(pop, ['setting', 'state'])
by_state = by_state[by_state.establishments >= 10]
by_state.to_csv('derived/rates-by-state.csv')
for s in ['hospital', 'nursing facility']:
    x = by_state.loc[s].sort_values('handling_rate', ascending=False)
    print(f'== {s}: handling rate per 100 FTE, highest and lowest states =='); print(x[['establishments', 'handling', 'handling_rate']].head(5)); print(x[['establishments', 'handling', 'handling_rate']].tail(5)); print()

== hospital: handling rate per 100 FTE, highest and lowest states ==
       establishments  handling  handling_rate
state                                         
MN                 97     843.0          1.263
ME                 33     237.0          1.229
OR                 50     341.0          0.888
WA                 94     783.0          0.851
AK                 10      58.0          0.835
       establishments  handling  handling_rate
state                                         
MT                 20      50.0          0.344
OH                201     472.0          0.249
WV                 45      86.0          0.240
LA                 66      65.0          0.142
TN                116     356.0          0.141

== nursing facility: handling rate per 100 FTE, highest and lowest states ==
       establishments  handling  handling_rate
state                                         
ME                 14      64.0          3.615
OR                 25      64.0          2.640
SD     

### By establishment size

In [8]:
pop['size_band'] = pd.cut(pop.employees, [99, 249, 499, 999, 10**7], labels=['100-249', '250-499', '500-999', '1,000+'])
by_size = rates(pop, ['setting', 'size_band'])
by_size.to_csv('derived/rates-by-size.csv'); by_size[['establishments', 'handling', 'handling_rate', 'lateral', 'lateral_rate']]

establishments  handling  handling_rate  lateral  lateral_rate
setting          size_band                                                                
hospital         100-249              1158     931.0          0.599     34.0         0.022
                 250-499               804    1232.0          0.548     73.0         0.032
                 500-999               632    2024.0          0.581    197.0         0.057
                 1,000+               1364   16168.0          0.530   1399.0         0.046
nursing facility 100-249              3235    4009.0          1.105     17.0         0.005
                 250-499               307     774.0          1.102      3.0         0.004
                 500-999                29     134.0          1.061      1.0         0.008
                 1,000+                 25      34.0          0.046      0.0         0.000

## 6. Lost-time cases

All handling and lateral-transfer cases in the case-detail file (not only the rate population), with at least one day away from work counted as lost time. Used by the cost estimate.

In [9]:
lost = cs.groupby('setting').apply(lambda g: pd.Series({
    'handling cases': int(g.handling.sum()), 'handling lost-time cases': int((g.handling & g.lost_time).sum()),
    'handling days away': int(g[g.handling].dafw.sum()), 'handling days restricted': int(g[g.handling].djtr.sum()),
    'lateral cases': int(g.lateral.sum()), 'lateral lost-time cases': int((g.lateral & g.lost_time).sum()),
    'lateral days away': int(g[g.lateral].dafw.sum()), 'lateral days restricted': int(g[g.lateral].djtr.sum())}), include_groups=False)
lost.to_csv('derived/lost-time.csv'); lost.T

setting,hospital,nursing facility
handling cases,20685,5654
handling lost-time cases,8473,2056
handling days away,312566,58178
handling days restricted,461884,125621
lateral cases,1726,22
lateral lost-time cases,743,4
lateral days away,28358,40
lateral days restricted,41597,498


## 7. Facility-level dataset

One row per establishment in the rate population: OSHA establishment ID, setting, state, size, hours, case counts, and rates. Nursing facilities carry their CMS Certification Number where the [`crosswalk`](../crosswalk/) links them (tiers A and B). No name or address is included.

In [10]:
XW = 'https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/main/crosswalk/osha-cms-crosswalk-nf-2025.csv'
xw = pd.read_csv(XW, dtype=str)
xw = xw[xw.tier.isin(['A', 'B'])][['osha_establishment_id', 'cms_ccn', 'tier']].rename(columns={'osha_establishment_id': 'establishment_id'})
fac = pop[['establishment_id', 'setting', 'state', 'employees', 'hours', 'cases', 'handling', 'lateral', 'handling_lost_time', 'lateral_lost_time']].copy()
for k in ['cases', 'handling', 'lateral', 'handling_lost_time', 'lateral_lost_time']: fac[k] = fac[k].astype(int)
fac['all_cases_rate'] = (fac.cases * 200000 / fac.hours).round(3)
fac['handling_rate'] = (fac.handling * 200000 / fac.hours).round(3)
fac = fac.merge(xw, on='establishment_id', how='left')
fac.to_csv('derived/facility-rates-2025.csv', index=False)
print(len(fac), 'establishments;', fac.cms_ccn.notna().sum(), 'nursing facilities with a CMS Certification Number')
fac.head()

7554 establishments; 3217 nursing facilities with a CMS Certification Number


,establishment_id,setting,state,employees,hours,cases,handling,lateral,handling_lost_time,lateral_lost_time,all_cases_rate,handling_rate,cms_ccn,tier
0,1004524,nursing facility,FL,182,315708,5,0,0,0,0,3.167,0.000,105652,A
1,1005125,hospital,NJ,1982,2635409,159,17,1,2,0,12.066,1.290,NaN,NaN
2,1005219,hospital,PA,274,417211,40,0,0,0,0,19.175,0.000,NaN,NaN
3,1005936,nursing facility,FL,394,645949,18,2,0,0,0,5.573,0.619,NaN,NaN
4,1006146,hospital,NC,335,512492,14,1,0,1,0,5.463,0.390,NaN,NaN


  adding: derived/ (stored 0%)
  adding: derived/rates-by-state.csv (deflated 60%)
  adding: derived/rates-by-setting.csv (deflated 46%)
  adding: derived/coverage.csv (deflated 20%)
  adding: derived/facility-rates-2025.csv (deflated 66%)
  adding: derived/rates-by-setting-no-hours-filter.csv (deflated 42%)
  adding: derived/lost-time.csv (deflated 45%)
  adding: derived/rates-by-size.csv (deflated 53%)
